# Phase 6 - Explainability

Do both models rely on drivers a credit officer would recognise and defend?

After what this project found about endogeneity, this is a substantive check. Two models,
two different failure modes:

- **GBM (champion, application-only)** is opaque. SHAP gives per-prediction attributions;
  aggregating them gives a global ranking, and correlating each feature against its own SHAP
  value gives the direction the model learned. A driver pointing the wrong way is a red flag
  no accuracy metric would ever surface.
- **Scorecard** is transparent but its coefficients are in WOE units, which nobody outside
  this repo reads. Section 4 converts them into a points table - what each answer on the
  application form is actually worth.

SHAP is run on the champion, not the full-pool model: there it would report that `sub_grade`
dominates, which is already known and explains nothing.


In [1]:
from pathlib import Path

import numpy as np
import polars as pl
import yaml

from credit_risk.data.ingestion import load_raw_accepted_loans
from credit_risk.data.target import build_target
from credit_risk.features.build_dataset import (
    APPLICATION_FEATURES, application_features, assemble_feature_matrix,
)
from credit_risk.models.gbm import prepare_lgb_frame, train_gbm
from credit_risk.models.scorecard import train_scorecard
from credit_risk.explainability.attribution import (
    compute_shap_values, points_range, rank_agreement, scorecard_points,
    shap_direction_report, shap_global_importance,
)

pl.Config.set_tbl_rows(80)
CONFIG_PATH = Path("../configs/base.yaml")
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")

final = assemble_feature_matrix(
    build_target(load_raw_accepted_loans(DATA_PATH), CONFIG_PATH), CONFIG_PATH
)
splits = {n: final.filter(pl.col("split") == n) for n in ("train", "validation", "oot_test")}
print({n: df.height for n, df in splits.items()})


{'train': 370443, 'validation': 421095, 'oot_test': 434407}


In [2]:
features = application_features(final)
params = yaml.safe_load(open("../configs/gbm_best_params_application.yaml"))
gbm, features = train_gbm(splits["train"], splits["validation"], params=params, features=features)
print(f"champion trained on {len(features)} features, {gbm.best_iteration} iterations")


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[659]	valid_0's auc: 0.706944
champion trained on 70 features, 659 iterations


## 1. What the GBM actually uses

SHAP values are computed on OOT, not train: the question is what drives the model on data it
has never seen. Values are in log-odds space, so they sum to the raw margin.

`share` is each feature's portion of total attribution. Watch for concentration - if two or
three features carry most of it, the other sixty are decoration and the model is simpler than
its feature count suggests.


In [3]:
oot_frame = prepare_lgb_frame(splits["oot_test"], features)
shap_values, sampled = compute_shap_values(gbm, oot_frame, sample_size=20_000)

importance = shap_global_importance(shap_values, sampled)
print(importance.head(25).to_pandas().to_string(index=False))
print(f"\ntop 5 carry {importance['share'].head(5).sum():.1%} of total attribution")
print(f"top 15 carry {importance['share'].head(15).sum():.1%}")
importance.write_csv("../docs/shap_importance.csv")


d:\Project_DS\credit-risk-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


                   feature  mean_abs_shap    share
                annual_inc       0.186940 0.070455
            fico_range_low       0.186518 0.070296
                 loan_amnt       0.169410 0.063848
      acc_open_past_24mths       0.152502 0.057476
                       dti       0.145012 0.054653
               term_months       0.133672 0.050379
        num_tl_op_past_12m       0.100348 0.037820
                   purpose       0.097805 0.036861
                addr_state       0.090869 0.034247
     mths_since_recent_inq       0.086827 0.032724
          percent_bc_gt_75       0.084516 0.031853
            home_ownership       0.072638 0.027376
      mths_since_recent_bc       0.068952 0.025987
      mo_sin_old_rev_tl_op       0.062243 0.023459
total_il_high_credit_limit       0.062223 0.023451
            pct_tl_nvr_dlq       0.055063 0.020752
           tot_hi_cred_lim       0.054936 0.020705
            bc_open_to_buy       0.053810 0.020280
            inq_last_6mths     

d:\Project_DS\credit-risk-system\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## 2. Direction — the check that matters

`direction` is Spearman between a feature's value and its own SHAP contribution.
`raises_risk = True` means higher values push predicted risk up.

Read the top features against what a credit officer would expect: higher `annual_inc` should
LOWER risk, higher `dti` should RAISE it, higher `fico_range_low` should LOWER it, more
recent inquiries should RAISE it. Any reversal among the high-importance features needs an
explanation before this model goes anywhere near serving.

Categoricals are reported as null rather than guessed: their codes have no order.


In [4]:
direction = shap_direction_report(shap_values, sampled)
review = (
    importance.join(direction, on="feature")
    .head(20)
    .select("feature", "mean_abs_shap", "share", "direction", "raises_risk")
)
print(review.to_pandas().to_string(index=False))
direction.write_csv("../docs/shap_direction.csv")


D:\Project_DS\credit-risk-system\src\credit_risk\explainability\attribution.py:68: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = float(spearmanr(column[mask], values[mask, i]).statistic)


                    feature  mean_abs_shap    share  direction raises_risk
                  loan_amnt       0.169410 0.063848   0.948340        True
                 emp_length       0.049736 0.018745        NaN        None
             home_ownership       0.072638 0.027376        NaN        None
                 annual_inc       0.186940 0.070455  -0.979708       False
        verification_status       0.045669 0.017212        NaN        None
                    purpose       0.097805 0.036861        NaN        None
                 addr_state       0.090869 0.034247        NaN        None
                        dti       0.145012 0.054653   0.979473        True
                delinq_2yrs       0.015189 0.005725   0.598982        True
             fico_range_low       0.186518 0.070296  -0.980956       False
             inq_last_6mths       0.053351 0.020107   0.860124        True
     mths_since_last_delinq       0.026339 0.009927  -0.905884       False
     mths_since_last_reco

In [5]:
# Explicit expectations, so the check is a test rather than an eyeball.
EXPECTED_RAISES_RISK = {
    "dti": True, "annual_inc": False, "fico_range_low": False,
    "mths_since_recent_inq": False,   # more months SINCE an inquiry is safer
    "acc_open_past_24mths": True, "percent_bc_gt_75": True,
    "term_months": True, "tot_hi_cred_lim": False, "bc_open_to_buy": False,
}
observed = dict(zip(*direction[["feature", "raises_risk"]]))
for feature, expected in EXPECTED_RAISES_RISK.items():
    got = observed.get(feature)
    verdict = "ok" if got == expected else ("UNKNOWN" if got is None else "CONTRADICTS EXPECTATION")
    print(f"{feature:<24} expected_raises_risk={str(expected):<5} observed={str(got):<5} {verdict}")


dti                      expected_raises_risk=True  observed=True  ok
annual_inc               expected_raises_risk=False observed=False ok
fico_range_low           expected_raises_risk=False observed=False ok
mths_since_recent_inq    expected_raises_risk=False observed=False ok
acc_open_past_24mths     expected_raises_risk=True  observed=True  ok
percent_bc_gt_75         expected_raises_risk=True  observed=True  ok
term_months              expected_raises_risk=True  observed=True  ok
tot_hi_cred_lim          expected_raises_risk=False observed=False ok
bc_open_to_buy           expected_raises_risk=False observed=False ok


## 3. SHAP versus IV — where the GBM earns its lift

IV is univariate; SHAP is not. Low agreement means the GBM is finding structure a
single-variable screen cannot see.

This is the expected explanation for a specific measured result: the GBM beats the linear
scorecard by +0.0122 AUC with `sub_grade` present but +0.0294 without it. With `sub_grade`
gone, the interactions have to be rediscovered, and apparently they are.


In [6]:
iv = pl.read_csv("../docs/iv_ranking_full.csv").filter(pl.col("iv").is_not_null())
print(rank_agreement(importance, iv))

# Features the GBM leans on that the IV screen rated as unusable (< 0.02).
weak_iv = set(iv.filter(pl.col("iv") < 0.02)["feature"].to_list())
missed = importance.head(20).filter(pl.col("feature").is_in(weak_iv))
print("\nhigh SHAP but screened out by IV:")
print(missed.to_pandas().to_string(index=False) if missed.height else "  none")


{'n_shared': 70, 'spearman': 0.8335403726708075}

high SHAP but screened out by IV:
                   feature  mean_abs_shap    share
                 loan_amnt       0.169410 0.063848
                addr_state       0.090869 0.034247
total_il_high_credit_limit       0.062223 0.023451
            pct_tl_nvr_dlq       0.055063 0.020752
                emp_length       0.049736 0.018745


## 4. The scorecard as a points table

This is the scorecard deliverable. Coefficients on WOE units are not usable by anyone; a
points lookup is.

    factor = pdo / ln(2),  offset = base_score - factor * ln(base_odds)
    points(feature, bin) = -factor * coefficient * WOE + (offset - factor * intercept) / n

Correctly signed coefficients are negative and higher WOE means safer, so safer bins earn
more points. Rows sum to the applicant's total score.


In [7]:
sc_model, sc_encoder = train_scorecard(splits["train"], features=APPLICATION_FEATURES)
points = scorecard_points(sc_encoder, sc_model, APPLICATION_FEATURES, pdo=20, base_score=600, base_odds=50.0)

print(points.filter(pl.col("feature").is_in(["fico_range_low", "dti", "term_months"]))
      .select("feature", "bin", "n", "bad_rate", "woe", "points")
      .to_pandas().to_string(index=False))
points.write_csv("../docs/scorecard_points.csv")


       feature            bin      n  bad_rate       woe  points
fico_range_low    (-inf, 660]  33474  0.120631 -0.336671    32.3
fico_range_low     (660, 665]  33559  0.114545 -0.278008    33.1
fico_range_low     (665, 670]  33906  0.111750 -0.250161    33.5
fico_range_low     (670, 675]  30120  0.105677 -0.187495    34.4
fico_range_low     (675, 680]  30223  0.104589 -0.175931    34.5
fico_range_low     (680, 685]  26828  0.095385 -0.073625    35.9
fico_range_low     (685, 690]  26201  0.092706 -0.042189    36.4
fico_range_low     (690, 695]  23636  0.083347  0.074446    38.0
fico_range_low     (695, 700]  20816  0.076480  0.167847    39.3
fico_range_low     (700, 710]  34712  0.072713  0.222533    40.0
fico_range_low     (710, 720]  25399  0.062168  0.390405    42.4
fico_range_low     (720, inf]  51569  0.046287  0.702258    46.7
           dti   (-inf, 5.35]  18537  0.063764  0.363245    42.7
           dti   (5.35, 9.18]  37131  0.064932  0.344048    42.4
           dti  (9.18, 11

## 5. Which features actually move a score

`swing` is the points difference between a feature's worst and best bin. A feature with a
large coefficient but bins that barely differ moves nobody's score, so this - not the
coefficient list - is the ranking to show a credit committee.

Compare it against the SHAP ranking from section 1. Broad agreement means both models read
the population the same way, which is the reassuring outcome. Sharp disagreement is worth
understanding before either is trusted.


In [8]:
swing = points_range(points)
print(swing.to_pandas().to_string(index=False))

print("\nagreement with SHAP ranking:", rank_agreement(
    importance, swing.rename({"swing": "value"})
))


              feature  min_points  max_points  n_bins  swing
              purpose        20.9        43.0      13   22.1
 acc_open_past_24mths        29.0        45.3       8   16.3
           annual_inc        29.5        45.5      11   16.0
mths_since_recent_inq        30.9        45.7      12   14.8
       fico_range_low        32.3        46.7      12   14.4
          term_months        28.4        41.1       2   12.7
                  dti        30.9        42.7      11   11.8
       bc_open_to_buy        32.4        43.9      14   11.5
 mo_sin_old_rev_tl_op        30.8        41.1      13   10.3
 mths_since_recent_bc        33.1        42.3      12    9.2
      tot_hi_cred_lim        34.4        42.2      10    7.8
  verification_status        34.7        42.3       3    7.6
     percent_bc_gt_75        34.5        40.8       8    6.3
       mo_sin_rcnt_tl        35.1        40.1      12    5.0
       home_ownership        34.3        39.3       4    5.0

agreement with SHAP ran

## 6. Findings to record

Into `docs/explainability_findings.md`:

| Question | Answer |
|---|---|
| Top 5 SHAP features and their share of attribution | |
| Any direction contradicting expectation? | |
| SHAP vs IV rank agreement | |
| Features with high SHAP but IV below 0.02 | |
| Largest points swing in the scorecard | |
| SHAP vs points-swing agreement | |

Limitations. SHAP direction is measured marginally, so a feature whose effect genuinely
reverses across its range (a U shape) averages toward zero and reads as no direction rather
than as a warning - a dependence plot is needed for anything that looks flat. Attribution is
computed on 20k sampled OOT rows; the global ranking is stable at that size but individual
tail features are not. And SHAP explains what the model does, never whether it is right:
a driver can be perfectly sensible and still be a proxy for something the lender may not
legally or ethically use.
